# 00 — Environment check

Cheap, runs anywhere. Verifies that the active environment can
actually see the things every later notebook will need: a working
Python, the right packages at pinned versions, a writable
Hugging Face cache, an HF auth token (if private models will be
loaded), and (on cluster) a CUDA-visible GPU.

Run this *first* on any new machine or container. If it fails,
every later notebook will fail in a more confusing way.

**Outputs.** A single `experiments/<run_dir>/environment_report.json`
plus `provenance.json`.

## Parameters

In [ ]:
run_name = 'env_check_local'
output_dir = f'experiments/{run_name}'
require_cuda = False    # flip to True on cluster
check_hf_auth = False   # flip to True if any private gated model is in scope
# Default = CPU-smokeable subset. Cluster sbatch wrapper overrides to
# the full GPU list (adds bitsandbytes, accelerate).
expected_packages = [
    'torch', 'transformers',
    'pandas', 'numpy', 'matplotlib',
    'papermill', 'jupyter',
]

## Setup

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import socket
import subprocess
import sys
import time

REPO = Path('.').resolve()
while REPO != REPO.parent and not (REPO / 'src').exists():
    REPO = REPO.parent
os.chdir(REPO)
print('repo root:', REPO)

## 1. Python and platform

In [ ]:
platform_info = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'host': socket.gethostname(),
    'cwd': str(REPO),
}
platform_info

## 2. Required packages

If any expected package is missing the cell raises rather than
silently warning. We want this notebook to be the loudest possible
failure surface.

In [ ]:
package_versions = {}
missing = []
for pkg in expected_packages:
    try:
        package_versions[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        missing.append(pkg)
if missing:
    raise RuntimeError(f'Missing required packages: {missing}')
package_versions

## 3. CUDA visibility (cluster only)

In [ ]:
import torch

cuda_info = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
    'device_names': [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count() if torch.cuda.is_available() else 0)
    ],
}
if require_cuda and not cuda_info['cuda_available']:
    raise RuntimeError('require_cuda=True but no GPU is visible')
cuda_info

## 4. Hugging Face cache and auth

In [ ]:
hf_home = os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
hf_cache_writable = os.access(Path(hf_home).parent, os.W_OK)
hf_info = {
    'hf_home': hf_home,
    'hf_cache_writable': hf_cache_writable,
    'hf_token_set': bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')),
}
if check_hf_auth and not hf_info['hf_token_set']:
    raise RuntimeError('check_hf_auth=True but no HF_TOKEN/HUGGING_FACE_HUB_TOKEN is set')
hf_info

## 5. Output dir is writable

In [ ]:
out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / '_writable.tmp').write_text('ok')
(out_dir / '_writable.tmp').unlink()
print('output_dir is writable:', out_dir)

## 6. Sanity check

Print the assembled report. If anything important is missing or
wrong, this is where it surfaces.

In [ ]:
report = {
    'platform': platform_info,
    'packages': package_versions,
    'cuda': cuda_info,
    'huggingface': hf_info,
}
(out_dir / 'environment_report.json').write_text(json.dumps(report, indent=2))
print('environment report:')
print(json.dumps(report, indent=2))

## 7. Provenance

In [ ]:
git_sha = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], capture_output=True, text=True
).stdout.strip() or 'unknown'
provenance = {
    'notebook': 'notebooks/00_environment_check.ipynb',
    'run_name': run_name,
    'output_dir': str(output_dir),
    'git_sha': git_sha,
    'host': socket.gethostname(),
    'slurm_job_id': os.environ.get('SLURM_JOB_ID'),
    'wallclock_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'parameters': {
        'run_name': run_name,
        'output_dir': output_dir,
        'require_cuda': require_cuda,
        'check_hf_auth': check_hf_auth,
    },
}
(out_dir / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print('wrote', out_dir / 'provenance.json')